In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from transformers import Blip2ForConditionalGeneration, Blip2Processor, AutoTokenizer
import easyocr
import ot

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"


image_folder  = "./MEME_UNIVOCI"
text_csv      = "./trascrizioni_ocr.csv"

#Commenta in base alla tipologia di modello fine-tunato che devi usare
#output_folder = "./SALIENCY_MAPS_MBLIP_LAST[-1]"
#output_folder = "./SALIENCY_MAPS_MBLIP_FULL[-1]"
#os.makedirs(output_folder, exist_ok=True)

#### Caricamento modello

In [ ]:
model_name = "Gregor/mblip-mt0-xl"
cache_dir  = "./cache"

print("Caricamento mBLIP:")
processor  = Blip2Processor.from_pretrained(model_name, cache_dir=cache_dir)
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    #device_map="auto",
    cache_dir=cache_dir
)
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
print("Modello caricato")

In [ ]:
class mBLIPClassifier(nn.Module):
    """
    Classificatore binario costruito sopra mBLIP.
    SI estrae un embedding visivo e uno testuale;
    due proiezioni li portano a dimensione comune, poi la concatenazione
    per una  classificazione lineare.
    """
    def __init__(self, blip_model, finetune=False):
        super().__init__()
        self.blip_model = blip_model
        self.image_proj = nn.Linear(768, 512) #768 è la hidden-size del Q-former
        self.text_proj  = nn.Linear(2048, 512) #2048 hidden-size dell'encoder testuale
        self.classifier = nn.Linear(1024, 1)

        #finetune=false allora è 'last layer fine tuned' else è 'full fine tuned'
        if not finetune:
            for param in self.blip_model.parameters():
                param.requires_grad = False

    def forward(self, pixel_values, input_ids, attention_mask):
        # patch immagine -> embedding
        vision_outputs = self.blip_model.vision_model(
            pixel_values=pixel_values, return_dict=True
        )
        image_embeds = vision_outputs.last_hidden_state

        image_attention_mask = torch.ones(
            image_embeds.size()[:-1],
            dtype=torch.long,
            device=image_embeds.device
        )
        #query tokens: vettori appresi che interrogano gli embedding visivi
        # ed estraggono un numero fisso di rappresentazioni rilevanti
        query_tokens = self.blip_model.query_tokens.expand(
            image_embeds.shape[0], -1, -1
        )
        qformer_outputs = self.blip_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True
        )
        #l'embedding visivo finale (media sui query tokens)
        image_embedding = qformer_outputs.last_hidden_state.mean(dim=1)
        
        #parte testuale
        text_outputs = self.blip_model.language_model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        text_embedding = text_outputs.last_hidden_state.mean(dim=1)

        #Normalizzazione L2 dei 2 embedding
        image_embedding = F.normalize(image_embedding, dim=-1)
        text_embedding  = F.normalize(text_embedding, dim=-1)

        #allineamento a fp16
        image_proj = self.image_proj(image_embedding.float())
        text_proj  = self.text_proj(text_embedding.float())
        #fusione delle due modalità
        combined   = torch.cat([image_proj, text_proj], dim=1)

        return self.classifier(combined)

#### Carica pesi


In [ ]:
model = mBLIPClassifier(blip_model, finetune=False).to(device)
model.load_state_dict(torch.load(
    "/scratch_share/mind/d.pizzo-thesis/mblip_last_layer.pt",
    map_location=device
))
#model.load_state_dict(torch.load(
#     "/scratch_share/mind/d.pizzo-thesis/mblip_full_fine.pt",
#     map_location=device
#))
model.eval()
print("Pesi caricati")

In [ ]:
activations = None #forward
gradients   = None #backward

def forward_hook(module, input, output):
    global activations
    # output puo essere una tupla,vogliamo il primo elemento che èl'hidden state
    if isinstance(output, tuple):
        activations = output[0].detach()
    else:
        activations = output.detach()

def backward_hook(module, grad_input, grad_output):
    global gradients
    # grad_output è una tupla — prendi il primo elemento non None
    for g in grad_output:
        if g is not None:
            gradients = g
            break

target_layer = model.blip_model.vision_model.encoder.layers[-35] #modifica in base al layer desiderato (-1 è l'ultimo; -39 è il primo)
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

In [ ]:
def compute_gradcam(image_tensor, text_input_ids, text_attention_mask):
    global activations, gradients
    
    model.blip_model.zero_grad() #azzero gradienti
    model.zero_grad()
    
    image_tensor = image_tensor.to(device)
    
    #forward pass completo attraverso il classificatore
    outputs = model(image_tensor, text_input_ids, text_attention_mask)
    
    #backward sull'output del classificatore
    outputs.sum().backward(retain_graph=True)
    
    print(f"Activations: {activations.shape if activations is not None else None}")
    print(f"Gradients: {gradients.shape if gradients is not None else None}")
    
    if activations is None or gradients is None:
        raise RuntimeError("Hook non triggerato")
    
    #rimuovo  token CLS
    patch_act  = activations[:, 1:, :].detach()
    patch_grad = gradients[:, 1:, :].detach()

    cam = patch_grad.abs().mean(dim=2).squeeze(0)  # [N patch]
    #normalizzazione
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    
    #rimappo vettore per potere fare poi confrotno con la ground truth umana
    grid_size = int(cam.shape[0] ** 0.5)

    cam = cam.reshape(grid_size, grid_size).cpu().float().numpy()
    cam = cv2.resize(cam, (768, 768))
    
    return cam, outputs.item()

In [ ]:
def integrated_gradients(text, tokenizer, image_embedding, steps=50):
    """
    IG per attribuire a ciascun token il suo contributo alla similarità imm-txt
    """
    print(f"IG per: {text[:30]}...")
    
    #fp32 per una migliore precisione
    model.blip_model.language_model.float()
    model.text_proj.float()
    model.image_proj.float()
    image_embedding = image_embedding.float()
    
    #tokenizzazione
    inputs = tokenizer(
        text, return_tensors="pt",
        padding=True, truncation=True
    ).to(device)
    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    #embedding
    input_embeds = model.blip_model.language_model.encoder\
        .embed_tokens(input_ids).float().detach().requires_grad_()

    #zero-baseline
    baseline = torch.zeros_like(input_embeds).to(device)

    grads = []
    #interpolazione in 50 step
    for alpha in torch.linspace(0, 1, steps):
        interpolated = baseline + alpha * (input_embeds - baseline)
        interpolated.retain_grad()

        #forward su encoder testuale
        outputs = model.blip_model.language_model.encoder(
            inputs_embeds=interpolated,
            attention_mask=attention_mask,
            return_dict=True
        )
        text_embedding = outputs.last_hidden_state.mean(dim=1)
        text_embedding = F.normalize(text_embedding, dim=-1)
         #proiezione e normalizzazione
        text_proj  = model.text_proj(text_embedding)
        image_proj = model.image_proj(image_embedding)
        text_proj  = F.normalize(text_proj, dim=-1)
        image_proj = F.normalize(image_proj, dim=-1)

        similarity = (text_proj * image_proj).sum()

        model.blip_model.zero_grad()
        similarity.backward()

        #scarta step con gradienti Nan o None
        grad = interpolated.grad
        if grad is not None and not torch.isnan(grad).any():
            grads.append(grad.detach().clone())
        else:
            print(f"  [SKIP] ={alpha:.2f}")
            grads.append(torch.zeros_like(input_embeds))

    #torno a fp16
    model.blip_model.language_model.half()

    avg_grads = torch.mean(torch.stack(grads), dim=0)
    #eventuali NaN residui nella media
    if torch.isnan(avg_grads).any():
        print("[WARN] avg_grads contiene nan")
        avg_grads = torch.nan_to_num(avg_grads, nan=0.0)

    #attribuisco IG
    attributions = (input_embeds - baseline) * avg_grads

    #relu
    token_importance = F.relu(attributions).norm(dim=-1).squeeze(0)\
        .detach().cpu().numpy()

    if token_importance.max() > 0:
        token_importance = (token_importance - token_importance.min()) / \
                           (token_importance.max() - token_importance.min() + 1e-8)

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))
    print(f"IG completato per: {text[:30]}...")
    return tokens, token_importance


##### Queste celle replicano ciò che veniva fatto nel framework di mCLIP

In [ ]:
def overlay_text_gradients_on_image(image, model_tokens, token_scores, reader,
                                     sigma=20, alpha=0.4, cmap_name='jet'):
    H, W = image.size[1], image.size[0]
    heatmap = np.zeros((H, W), dtype=np.float32)

    ocr_results = reader.readtext(np.array(image))

    scores_norm = (token_scores - token_scores.min()) / \
                  (token_scores.max() - token_scores.min() + 1e-8)

    for bbox, ocr_text, _ in ocr_results:
        ocr_subtokens = tokenizer.tokenize(ocr_text)
        N = len(ocr_subtokens)
        if N == 0:
            continue

        bbox = np.array(bbox)
        xmin, ymin = bbox[:, 0].min(), bbox[:, 1].min()
        xmax, ymax = bbox[:, 0].max(), bbox[:, 1].max()
        width_per_token = (xmax - xmin) / N

        for idx, ocr_token in enumerate(ocr_subtokens):
            clean_ocr = ocr_token.replace("▁", "").lower()
            score = 0
            for i, model_token in enumerate(model_tokens):
                clean_model = model_token.replace("▁", "").lower()
                #if clean_ocr == clean_model:
                if clean_ocr in clean_model or clean_model in clean_ocr:
                    score = scores_norm[i]
                    print(f"MATCH: ocr='{clean_ocr}' model='{clean_model}' score={score:.3f}")
                    break

            x0 = int(xmin + idx * width_per_token)
            x1 = int(xmin + (idx + 1) * width_per_token)
            token_bbox = np.array([[x0, ymin], [x1, ymin],
                                   [x1, ymax], [x0, ymax]])

            mask = np.zeros_like(heatmap, dtype=np.uint8)
            cv2.fillPoly(mask, [token_bbox.astype(np.int32)], 1)
            heatmap += mask * score

    heatmap_norm = heatmap / (heatmap.max() + 1e-8)
    cmap = matplotlib.colormaps.get_cmap(cmap_name)
    overlay_map = (cmap(heatmap_norm)[..., :3] * 255).astype(np.uint8)
    blended = cv2.addWeighted(np.array(image), 1 - alpha, overlay_map, alpha, 0)

    return Image.fromarray(blended), heatmap

In [ ]:
def show_cam_and_ig_on_image(img, combined, meme_name):
    plt.figure(figsize=(8, 8), dpi=96)
    plt.imshow(img)
    plt.imshow(combined, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    save_path = os.path.join(output_folder, f"{meme_name}_gradcam_and_ig.jpg")
    plt.savefig(save_path)

In [ ]:
text_df = pd.read_csv(text_csv)
ids, texts = [], []
for i in range(len(text_df)):
    ids.append(text_df["file_name"][i])
    texts.append(text_df["full_text"][i])

In [ ]:
reader = easyocr.Reader(['it', 'en'], gpu=torch.cuda.is_available())

In [ ]:
#main loop-la logica è identica a clip
#si fonde il contributo di Grad-Cam con quello di IG
for doc_id, text in zip(ids, texts):
    print(f"\nElaboro: {doc_id}")

    #carica e preprocessing immagine
    img_path = os.path.join(image_folder, doc_id)
    image    = Image.open(img_path).convert("RGB")
    inputs       = processor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device).half()

    meme_name = doc_id.replace(".jpg", "")

    #e,bedding visivo
    with torch.no_grad():
        vision_out  = model.blip_model.vision_model(
            pixel_values=pixel_values, return_dict=True
        )
        image_embeds = vision_out.last_hidden_state

        image_attn_mask = torch.ones(
            image_embeds.size()[:-1],
            dtype=torch.long, device=device
        )
        query_tokens = model.blip_model.query_tokens.expand(
            image_embeds.shape[0], -1, -1
        )
        qformer_out = model.blip_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn_mask,
            return_dict=True
        )
        image_embedding = qformer_out.last_hidden_state.mean(dim=1)
        image_embedding = F.normalize(image_embedding.float(), dim=-1)

    #IG
    tokens, token_scores = integrated_gradients(
        text, tokenizer, image_embedding, steps=50
    )

    #tresholding: azzero token sotto 0.3 di importanza,per tenere 
    #i maggiori contributi scartando il rumore di fondo
    token_scores[token_scores < 0.3] = 0 #regolarizzazione IG

    #proiezione dei contributi sul testo del meme
    _, heatmap_ig = overlay_text_gradients_on_image(
        image, tokens, token_scores, reader, cmap_name='jet'
    )

    #heatmap_ig_norm = heatmap_ig / (heatmap_ig.max() + 1e-8)
    if heatmap_ig.max() > 0:
        heatmap_ig_norm = heatmap_ig / heatmap_ig.max()
    else:
        heatmap_ig_norm = np.zeros_like(heatmap_ig)
        print("heatmap_ig nulla,nessun testo trovato dall' OCR")

    # tokenizza testo per gradcam
    text_inputs = tokenizer(
        text, return_tensors="pt",
        padding=True, truncation=True
    ).to(device)

    #abilita gradienti del ViT(richiesti da gradcam)
    for param in model.blip_model.vision_model.parameters():
        param.requires_grad = True

    cam, sim = compute_gradcam(
        pixel_values,
        text_inputs["input_ids"],
        text_inputs["attention_mask"]
    )
    cam_norm = cam / (cam.max() + 1e-8)
    #li ricongelo
    for param in model.blip_model.vision_model.parameters():
        param.requires_grad = False

    #fusione delle parti testuali e visive
    #pesi 70%/30%,scelti sperimentalmente in base agli output ottenuti
    # cercando di avere il miglior allineamento umano
    #combined_matrix = cam_norm + heatmap_ig_norm
    combined_matrix = 0.7 * cam_norm + 0.3 * heatmap_ig_norm

    #controlla nan
    if np.isnan(combined_matrix).any():
        print("La combined_matrix contiene nan")
        combined_matrix = cam_norm.copy()
    
    #smoothing e normalizzazione
    combined_matrix = gaussian_filter(combined_matrix, sigma=20)
    combined_matrix = combined_matrix / (combined_matrix.max() + 1e-8)

    #salva immagine e file .npy
    show_cam_and_ig_on_image(image, combined_matrix, meme_name)
    np.save(
        os.path.join(output_folder, f"{meme_name}_gradcam_and_ig.npy"),
        combined_matrix
    )
    print(f"Salvato: {meme_name}")

#### COntrollo migliori e peggiori meme

In [ ]:
csv_path = "/scratch_share/mind/d.pizzo-thesis/metrics_mblip_last[-20].csv"
df = pd.read_csv(csv_path)

print("Colonne:", df.columns.tolist())
print(f"Righe: {len(df)}\n")
meme_col = 'meme_name'

best_nss  = df.loc[df['NSS'].idxmax()]
worst_nss = df.loc[df['NSS'].idxmin()]
print("NSS  (alto = meglio)")
print(f"  Migliore: {best_nss[meme_col]}  →  NSS = {best_nss['NSS']:.4f}")
print(f"  Peggiore: {worst_nss[meme_col]}  →  NSS = {worst_nss['NSS']:.4f}\n")

best_emd  = df.loc[df['EMD'].idxmin()]
worst_emd = df.loc[df['EMD'].idxmax()]
print("EMD  (basso = meglio)")
print(f"  Migliore: {best_emd[meme_col]}  →  EMD = {best_emd['EMD']:.4f}")
print(f"  Peggiore: {worst_emd[meme_col]}  →  EMD = {worst_emd['EMD']:.4f}")

### ANALISI HEADS

In [ ]:
#costruzione della ground truth umana,identica alla aprte di mCLIP
def set_values():

    screen_width, screen_height = 1920, 1080
    meme_width, meme_height = 768, 768
    offset_x = (screen_width - meme_width) // 2
    offset_y = (screen_height - meme_height) // 2
    heatmap = np.zeros((meme_height, meme_width), dtype=float)
    return offset_x, offset_y, heatmap

def set_heatmap(df, offset_x, offset_y, base_heatmap):

    heatmap = base_heatmap.copy()
    for _, row in df.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        duration = row['CURRENT_FIX_DURATION']
        if 0 <= x < 768 and 0 <= y < 768:
            heatmap[y, x] += duration
    heatmap = gaussian_filter(heatmap, sigma=20)
    return heatmap

### Calcolo metriche identico al framework di mCLIP

In [ ]:
def NSS(pred_map, gt_fix):
    pred_z = (pred_map - np.mean(pred_map)) / (np.std(pred_map) + 1e-8)
    return np.mean(pred_z[gt_fix > 0])

def EMD_2d(pred_map, gt_map, size=64):
    cam_small    = cv2.resize(pred_map, (size, size))
    gt_map_small = cv2.resize(gt_map,   (size, size))

    a = cam_small.ravel().astype(np.float64)
    b = gt_map_small.ravel().astype(np.float64)
    a /= (a.sum() + 1e-8)
    b /= (b.sum() + 1e-8)

    x, y = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    coords = np.stack([x.ravel(), y.ravel()], axis=1)
    M = ot.dist(coords, coords, metric="euclidean")

    return ot.emd2(a, b, M)

In [ ]:
def get_head_heatmaps_mblip(image_tensor, layer_idx):
    """
    Estrae mappe di attention per singola head da un layer dell'image
    encoder di mBLIP. L'attention di BLIP usa una proiezione QueryKeyValue fusa quindi
    è ricostruita poi a mano,solo così posso ottenere i pesi separati per head
    """
    captured = {}
    def hook(module, args, kwargs):
        if len(args) > 0:
            captured['x'] = args[0].detach()
        else:
            captured['x'] = kwargs['hidden_states'].detach()
    #self-attention del layer scelto dell'encoder visivo
    target = model.blip_model.vision_model.encoder.layers[layer_idx].self_attn
    h = target.register_forward_pre_hook(hook, with_kwargs=True)
    #forward
    with torch.no_grad():
        _ = model.blip_model.vision_model(pixel_values=image_tensor, return_dict=True)
    h.remove()

    #costruzione dell'attention per head
    x = captured['x']
    bsz, T, embed_dim = x.size()
    n_heads = target.num_heads
    scale   = target.scale
    #proiezione Query,Key,Value
    mixed_qkv = target.qkv(x)
    mixed_qkv = mixed_qkv.reshape(bsz, T, 3, n_heads, embed_dim // n_heads)
    mixed_qkv = mixed_qkv.permute(2, 0, 3, 1, 4)
    q, k, _ = mixed_qkv[0], mixed_qkv[1], mixed_qkv[2]
    #dot-product per head
    attn = torch.matmul(q, k.transpose(-1, -2)) * scale
    attn = attn.softmax(dim=-1)
    cls_attn = attn[0, :, 0, 1:]
    n_patches = cls_attn.shape[1]
    grid = int(n_patches ** 0.5)
    #ogni head ritorna come griglia spaziale
    head_heatmaps = cls_attn.reshape(n_heads, grid, grid).cpu().float().numpy()
    return head_heatmaps

#setup da modificare in  base all'analisi da fare
LAYER_IDX = -39 #indici dal fondo
model_tag = "fine"      

out_dir = "/scratch_share/mind/d.pizzo-thesis/head_attention_mblip"
os.makedirs(out_dir, exist_ok=True)
df_meme_agg_folder = "/scratch_share/mind/d.pizzo-thesis/DF_MEME_AGGREGATED"

#pulizia hook per essere sicuro che non ci siano interferenze
for _l in model.blip_model.vision_model.encoder.layers:
    _l.self_attn._forward_pre_hooks.clear()
    _l.self_attn._forward_hooks.clear()
    _l.self_attn._backward_hooks.clear()

reader = easyocr.Reader(['it','en'], gpu=torch.cuda.is_available())
offset_x, offset_y, base_heatmap = set_values()


print(f"\n LAYER {LAYER_IDX}  ({model_tag})")
results = []

#loop per ogni meme sulle 16 head
for ci, (doc_id, text) in enumerate(zip(ids, texts), start=1):
    img_path = os.path.join(image_folder, doc_id)
    image = Image.open(img_path).convert("RGB")

    # heatmap umana
    meme_aggregated = pd.read_csv(os.path.join(df_meme_agg_folder, f"{doc_id}.csv"))
    human_heatmap = set_heatmap(meme_aggregated, offset_x, offset_y, base_heatmap)
    gt_fix = np.zeros((768, 768))
    for _, row in meme_aggregated.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        if 0 <= x < 768 and 0 <= y < 768:
            gt_fix[y, x] = 1
    gt_map = human_heatmap.copy()
    gt_map = (gt_map - gt_map.min()) / (gt_map.max() + 1e-8)

    #preprocessing immagine mBLIP
    pv = processor(images=image, return_tensors="pt")['pixel_values'].to(device).half()

    # embedding visivo per IG
    with torch.no_grad():
        vision_out = model.blip_model.vision_model(pixel_values=pv, return_dict=True)
        image_embeds = vision_out.last_hidden_state
        image_attn_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=device)
        query_tokens = model.blip_model.query_tokens.expand(image_embeds.shape[0], -1, -1)
        qformer_out = model.blip_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn_mask,
            return_dict=True
        )
        image_embedding = qformer_out.last_hidden_state.mean(dim=1)
        image_embedding = F.normalize(image_embedding.float(), dim=-1)

    #IG
    tokens, token_scores = integrated_gradients(text, tokenizer, image_embedding, steps=50)
    token_scores[token_scores < 0.3] = 0
    _, heatmap_ig = overlay_text_gradients_on_image(image, tokens, token_scores, reader, cmap_name='jet')
    if heatmap_ig.max() > 0:
        heatmap_ig_norm = heatmap_ig / heatmap_ig.max()
    else:
        heatmap_ig_norm = np.zeros_like(heatmap_ig)

    # attention delle 16 head del layer scelto
    head_heatmaps = get_head_heatmaps_mblip(pv, LAYER_IDX)   # [16, 16, 16]

    #metriche epr ogni head
    for head_idx in range(head_heatmaps.shape[0]):
        hv = head_heatmaps[head_idx]
        hv = (hv - hv.min()) / (hv.max() - hv.min() + 1e-8)
        hv = cv2.resize(hv, (768, 768))
        combined = hv + heatmap_ig_norm
        combined = gaussian_filter(combined, sigma=20)
        combined = combined / (combined.max() + 1e-8)
        results.append({
            'meme': doc_id,
            'head': head_idx,
            'NSS': NSS(combined, gt_fix),
            'EMD': EMD_2d(combined, gt_map)
        })

    print(f"  meme {ci}/{len(ids)}")

#csv
df_results = pd.DataFrame(results)
csv_path = os.path.join(out_dir, f"metrics_heads_mblip_{model_tag}[{LAYER_IDX}].csv")
df_results.to_csv(csv_path, index=False)
print(f"\nSalvato: {csv_path}  ({len(df_results)} righe)")

## Meme della migliore combinazione (in base EMD)

In [ ]:
import os, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# combinazione scelta

LAYER_IDX = -28      # indice neg
HEAD_IDX  = 7        # le head sono numerate da 0 invece che da 1 !!!!!
model_tag = "full"

out_dir = f"/scratch_share/mind/d.pizzo-thesis/head_attention_mblip/HEATMAPS_MBLIP_{model_tag}_L[{LAYER_IDX}]_H{HEAD_IDX+1}"
os.makedirs(out_dir, exist_ok=True)

# pulizia hook
for _l in model.blip_model.vision_model.encoder.layers:
    _l.self_attn._forward_pre_hooks.clear()
    _l.self_attn._forward_hooks.clear()
    _l.self_attn._backward_hooks.clear()

reader = easyocr.Reader(['it','en'], gpu=torch.cuda.is_available())

# salvo le heatmap dei 96 meme, processo simile a prima
contatore = 0
for doc_id, text in zip(ids, texts):
    img_path = os.path.join(image_folder, doc_id)
    image = Image.open(img_path).convert("RGB")
    meme_name = doc_id.replace(".jpg", "")

    # preprocessing
    pv = processor(images=image, return_tensors="pt")['pixel_values'].to(device).half()

    # visual embedding
    with torch.no_grad():
        vision_out = model.blip_model.vision_model(pixel_values=pv, return_dict=True)
        image_embeds = vision_out.last_hidden_state
        image_attn_mask = torch.ones(image_embeds.size()[:-1], dtype=torch.long, device=device)
        query_tokens = model.blip_model.query_tokens.expand(image_embeds.shape[0], -1, -1)
        qformer_out = model.blip_model.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attn_mask,
            return_dict=True
        )
        image_embedding = qformer_out.last_hidden_state.mean(dim=1)
        image_embedding = F.normalize(image_embedding.float(), dim=-1)

    # Ig
    tokens, token_scores = integrated_gradients(text, tokenizer, image_embedding, steps=50)
    token_scores[token_scores < 0.3] = 0
    _, heatmap_ig = overlay_text_gradients_on_image(image, tokens, token_scores, reader, cmap_name='jet')
    if heatmap_ig.max() > 0:
        heatmap_ig_norm = heatmap_ig / heatmap_ig.max()
    else:
        heatmap_ig_norm = np.zeros_like(heatmap_ig)

    # -Attention della specifica head
    head_heatmaps = get_head_heatmaps_mblip(pv, LAYER_IDX)   # [16, 16, 16]
    hv = head_heatmaps[HEAD_IDX]
    hv = (hv - hv.min()) / (hv.max() - hv.min() + 1e-8)
    hv = cv2.resize(hv, (768, 768))

    #Fusione identica a prima
    combined = hv + heatmap_ig_norm
    combined = gaussian_filter(combined, sigma=20)
    combined = combined / (combined.max() + 1e-8)

    # save
    image_768 = image.resize((768, 768))
    plt.figure(figsize=(8, 8))
    plt.imshow(image_768)
    plt.imshow(combined, cmap='jet', alpha=0.4)
    plt.axis('off')
    plt.title(f"{meme_name} — Layer {LAYER_IDX} · Head {HEAD_IDX+1}", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{meme_name}_L[{LAYER_IDX}]_H{HEAD_IDX+1}.png"),
                dpi=150, bbox_inches='tight')
    plt.close()

    contatore += 1
    print(f"Meme elaborati: {contatore}/{len(ids)}")

print(f"\nFatto! {contatore} heatmap salvate in {out_dir}")